# SBND RWM Timing Calibration: Apply It Step by Step

- Applies the calibration CSV `sbnd_fine_calibration.csv` (one row per time chunk) to every entry of the uncalibrated `TimingCalibAna` ntuple of one run
- Output per entry: `calib_chunk` (CSV row), `calib_bucket`, `calib_bucket_pos_ns`, `calib_time_ns`, `calib_status`

## 1. Setup

- Set the paths in the Setup cell: `NTUPLE` = ntuple per run, `CALIB_DIR` = calibration notebook output folder; `RUN` = 1 or 2

In [ ]:
import ast
import os
import time
os.environ['TZ'] = 'UTC'
time.tzset()

import numpy as np
import pandas as pd
import uproot
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcdefaults()

import sys
sys.path.insert(0, '../lib')
from stacking import stacked_bunch_fit           # figures only
from plot_helpers import SEG_COLORS, draw_asym_fit
import event_timestamps_run1 as ev1
import event_timestamps_run2 as ev2

RUN = 1   # 1 or 2
ev = {1: ev1, 2: ev2}[RUN]   # run registry: segment names (figures)
# ===== Set these paths =====
NTUPLE = {1: '/path/to/run1_timing_calib_ntuple.root',   # input ntuple per run (README: Data on Fermilab)
          2: '/path/to/run2_timing_calib_ntuple.root'}[RUN]
CALIB_DIR = '../output'   # calibration notebook output folder: CALIB_DIR/run{1,2}/sbnd_fine_calibration.csv
CALIB_CSV = f'{CALIB_DIR}/run{RUN}/sbnd_fine_calibration.csv'
BAD_TIME = -9000   # opflash_t0_corrected_ns ≤ this = no corrected time (−9999)
print(f'ntuple:          {NTUPLE}\ncalibration CSV: {CALIB_CSV}')

## 2. Read the Entries (Uncalibrated Ntuple)

Tree `timingcalib/timingCalibTree`, one row per entry (ν-candidate slice); all times uncalibrated:

| Ntuple branch | Used for |
|---|---|
| `run`, `subrun`, `event`, `slice_id` | entry key |
| `corr_flash_status` | selection: 0 = entry has a corrected OpFlash |
| `tdc_rwm_ns` | selection: valid RWM timestamp (≠ 0, ≠ max uint64); `ts` = absolute event time → chunk |
| `opflash_t0_corrected_ns` | flash time [ns], RWM-referenced (≤ −9000 = not set → `bad_time`) |
| `nu_tof_charge_ns` | ν time of flight from the detector front face to the charge vertex [ns] |

- `uncalib_time_ns` = `opflash_t0_corrected_ns` − `nu_tof_charge_ns` (time at the detector front face, z = 0) = the time the calibration corrects

In [ ]:
BRANCHES = ['run', 'subrun', 'event', 'slice_id', 'corr_flash_status', 'tdc_rwm_ns', 'opflash_t0_corrected_ns', 'nu_tof_charge_ns']
parts = []
for a in uproot.open(NTUPLE)['timingcalib/timingCalibTree'].iterate(BRANCHES, library='pd', step_size='200 MB'):
    good = (a['corr_flash_status'] == 0) & (a['tdc_rwm_ns'] != 0) & (a['tdc_rwm_ns'] != np.iinfo(np.uint64).max)
    parts.append(a[good])
df = pd.concat(parts, ignore_index=True).drop(columns='corr_flash_status')
del parts
df['uncalib_time_ns'] = df['opflash_t0_corrected_ns'] - df['nu_tof_charge_ns']   # time at the detector front face (z = 0)
df['ts'] = pd.to_datetime(df['tdc_rwm_ns'], unit='ns')                           # absolute event time (RWM timestamp)
print(f'{NTUPLE} → timingcalib/timingCalibTree, branches {BRANCHES}')
print(f'Run {RUN}: {len(df)} entries with a corrected OpFlash and a good RWM timestamp, runs {df.run.min()}-{df.run.max()}')

## 3. Read the Calibration Table

| CSV column | Used for |
|---|---|
| `run`, `start`, `end` | match an entry to its row: event run in `run`, event RWM timestamp in [`start`, `end`] (step 4) |
| `skip_reason` | non-empty = chunk without constants (step 4) |
| `peak_positions` | positions of the 81 core buckets [ns] (step 5) |
| `edge_index`, `edge_positions` | edge buckets (partly filled, up to 2 per side): bucket index k (−2, −1 / 81, 82) and position [ns] (step 5) |
| `window_offset` | bucket number = fit index (or edge index) + `window_offset`: same number = same RF bucket in every chunk (step 5) |
| `period` | bucket spacing: spill window = first position − `period` / 2 … last position + `period` / 2 (step 5) |
| `intercept`, `segment` | figures only: position of bucket 0, stacking segment |

In [ ]:
USE_COLS = ['run', 'start', 'end', 'skip_reason', 'period', 'peak_positions', 'edge_index', 'edge_positions', 'window_offset',
            'intercept', 'segment']
calib = pd.read_csv(CALIB_CSV, usecols=USE_COLS)[USE_COLS]
calib['start'] = pd.to_datetime(calib['start'])
calib['end'] = pd.to_datetime(calib['end'])
calib['skip_reason'] = calib['skip_reason'].fillna('')
for col in ('run', 'peak_positions', 'edge_index', 'edge_positions'):   # lists stored as text, e.g. '[18388, 18389]'
    calib[col] = calib[col].apply(ast.literal_eval)
fitted = calib['skip_reason'] == ''
print(f'{CALIB_CSV}: {len(calib)} chunks ({fitted.sum()} fitted, {(~fitted).sum()} skipped); calib_chunk = row number in this file')
calib.head(3)

## 4. Match Each Uncalibrated Entry to the Calibration CSV by Its Event RWM Timestamp

- Match: CSV row with the entry's run in `run` and its event RWM timestamp `ts` (`tdc_rwm_ns`) in [`start`, `end`] → `calib_chunk` = CSV row number
- No row → `no_chunk`; row without constants (`skip_reason` set) → `skipped_chunk`
- The matched row gives the bucket positions (`peak_positions` + `edge_positions`) used in step 5

In [ ]:
calib_chunk = np.full(len(df), -1)
run, ts = df['run'].values, df['ts'].values
for i, row in calib.iterrows():
    m = np.isin(run, row['run']) & (ts >= row['start'].to_datetime64()) & (ts <= row['end'].to_datetime64())
    assert (calib_chunk[m] < 0).all(), f'entry matched to two rows (row {i})'
    calib_chunk[m] = i
df['calib_chunk'] = calib_chunk
in_chunk = calib_chunk >= 0
skipped_chunk = in_chunk & ~fitted.values[np.maximum(calib_chunk, 0)]
print(f'{len(df):,} entries: {(in_chunk & ~skipped_chunk).sum():,} matched to a CSV row with constants, '
      f'{skipped_chunk.sum():,} to a row without constants, {(~in_chunk).sum():,} unmatched')


def bucket_comb(row):
    """One CSV row → (bucket numbers, bucket positions [ns]), in time order; bucket number = fit index (or edge index) + window_offset."""
    core = np.arange(len(row['peak_positions']))
    buckets = np.concatenate([core, row['edge_index']]).astype(int) + int(row['window_offset'])
    pos = np.concatenate([row['peak_positions'], row['edge_positions']])
    o = np.argsort(pos)
    return buckets[o], pos[o]


EX_ROW = int(pd.Series(calib_chunk[in_chunk & ~skipped_chunk]).value_counts().index[0])   # example: CSV row with the most entries
ex = calib.loc[EX_ROW]
ex_buckets, ex_pos = bucket_comb(ex)
print(f"example: CSV row {EX_ROW} (runs {ex['run']}, {ex['start']:%Y-%m-%d %H:%M} → {ex['end']:%Y-%m-%d %H:%M}) "
      f"matches {(calib_chunk == EX_ROW).sum():,} entries; its {len(ex_pos)} bucket positions:")
pd.DataFrame({'bucket': ex_buckets, 'position_ns': ex_pos}).iloc[[0, 1, 2, -3, -2, -1]]

## 5. Calibrated Time of Each Entry

- A BNB spill is 81 filled bunches ("core buckets") 18.9 ns apart, plus up to 2 partly filled "edge buckets" per side; the matched CSV row gives the position of each core and edge bucket for that period of data taking
- Each entry goes to its nearest bucket position: `calib_bucket` (bucket number), `calib_bucket_pos_ns` (its position); calibrated time `calib_time_ns` = `uncalib_time_ns` − `calib_bucket_pos_ns` (0 = on the bucket position)
- In spill = from half a bucket spacing (`period` / 2) before the first to `period` / 2 after the last bucket, edge buckets included; entries outside are not calibrated

In [ ]:
df['calib_status'] = np.where(in_chunk, 'skipped_chunk', 'no_chunk').astype(object)
for col in ('calib_bucket', 'calib_bucket_pos_ns', 'calib_time_ns'):
    df[col] = np.nan
t, bad = df['uncalib_time_ns'].values, df['opflash_t0_corrected_ns'].values <= BAD_TIME
for i, row in calib[fitted].iterrows():
    m = calib_chunk == i
    df.loc[m & bad, 'calib_status'] = 'bad_time'
    g = np.flatnonzero(m & ~bad)
    buckets, pos = bucket_comb(row)
    lo, hi = pos[0] - row['period'] / 2, pos[-1] + row['period'] / 2      # spill window
    j = np.clip(np.searchsorted(pos, t[g]), 1, len(pos) - 1)
    k = np.where(np.abs(t[g] - pos[j - 1]) <= np.abs(t[g] - pos[j]), j - 1, j)   # nearest bucket (tie → earlier)
    inside = (t[g] >= lo) & (t[g] < hi)
    df.loc[g, 'calib_status'] = np.where(inside, 'in_spill', np.where(t[g] < lo, 'before_spill', 'after_spill'))
    gi = g[inside]
    df.loc[gi, 'calib_bucket'] = buckets[k[inside]]
    df.loc[gi, 'calib_bucket_pos_ns'] = pos[k[inside]]
    df.loc[gi, 'calib_time_ns'] = t[gi] - pos[k[inside]]
df['calib_bucket'] = df['calib_bucket'].astype('Int64')
n_cal = (df['calib_status'] == 'in_spill').sum()
print(f'{n_cal:,} of {len(df):,} entries calibrated ({100 * n_cal / len(df):.1f} %); the others: summary in step 7')

## 6. Example Entry

- First in-spill entry of the example CSV row (step 4), step by step

In [ ]:
s = df[(df.calib_chunk == EX_ROW) & (df.calib_status == 'in_spill')].iloc[0]
pd.Series({
    'ntuple: opflash_t0_corrected_ns': s.opflash_t0_corrected_ns,
    'ntuple: nu_tof_charge_ns': s.nu_tof_charge_ns,
    'uncalib_time_ns = opflash − tof': s.uncalib_time_ns,
    'ntuple: tdc_rwm_ns → ts': f'{s.ts:%Y-%m-%d %H:%M:%S.%f}',
    'calib_chunk (CSV row: run, start ≤ ts ≤ end)': f"{EX_ROW} ({ex['run']}, {ex['start']:%m-%d %H:%M} → {ex['end']:%m-%d %H:%M})",
    'calib_bucket (CSV: nearest of peak / edge positions)': int(s.calib_bucket),
    'calib_bucket_pos_ns': s.calib_bucket_pos_ns,
    'calib_time_ns = uncalib_time_ns − calib_bucket_pos_ns': s.calib_time_ns,
}, name='example entry').to_frame()

In [ ]:
in_ex = df.calib_chunk == EX_ROW
lo, hi = ex_pos[0] - ex['period'] / 2, ex_pos[-1] + ex['period'] / 2
fig, ax = plt.subplots(1, 2, figsize=(13.2, 4.4))
ax[0].hist(df.uncalib_time_ns[in_ex], bins=np.arange(lo - 100, hi + 100, 2), histtype='step', color='tab:blue')
for x in (lo, hi):
    ax[0].axvline(x, color='red', ls='--', lw=1)
ax[0].axvline(s.uncalib_time_ns, color='black', lw=1.5)
ax[0].set(xlabel='uncalib_time_ns [ns]', ylabel='Entries / [2 ns]', title=f'Chunk {EX_ROW}: Spill Window (red)')

z_lo, z_hi = s.calib_bucket_pos_ns - 1.5 * ex['period'], s.calib_bucket_pos_ns + 1.5 * ex['period']
ax[1].hist(df.uncalib_time_ns[in_ex], bins=np.arange(z_lo, z_hi, 0.25), histtype='step', color='tab:blue')
for x in ex_pos[(ex_pos > z_lo) & (ex_pos < z_hi)]:
    ax[1].axvline(x, color='red', ls='--', lw=1)
for x in ((ex_pos[1:] + ex_pos[:-1]) / 2)[(ex_pos[1:] > z_lo) & (ex_pos[:-1] < z_hi)]:
    ax[1].axvline(x, color='grey', ls=':', lw=1)
ax[1].axvline(s.uncalib_time_ns, color='black', lw=1.5)
y1 = ax[1].get_ylim()[1]
ax[1].set_ylim(0, 1.3 * y1)   # headroom for the annotation
ax[1].annotate('', (s.uncalib_time_ns, 1.1 * y1), (s.calib_bucket_pos_ns, 1.1 * y1), arrowprops=dict(arrowstyle='->', lw=1.2))
ax[1].text(s.calib_bucket_pos_ns + 1, 1.1 * y1, f'calib_time_ns = {s.calib_time_ns:+.2f} ns', va='center')
ax[1].set(xlabel='uncalib_time_ns [ns]', ylabel='Entries / [0.25 ns]',
          title=f'Bucket {int(s.calib_bucket)}: positions (red), boundaries (grey), example entry (black)')
plt.tight_layout()
plt.show()

## 7. Summary

- Every entry gets one `calib_status`; only `in_spill` entries have a calibrated time

In [ ]:
MEANING = {
    'in_spill': 'calibrated: nearest bucket found, calib_time_ns set',
    'before_spill': 'earlier than the spill (more than period / 2 before the first bucket), not calibrated',
    'after_spill': 'later than the spill (more than period / 2 after the last bucket), not calibrated',
    'skipped_chunk': 'matched CSV row has no constants (too few entries or fit failed), not calibrated',
    'bad_time': 'no corrected OpFlash time in the ntuple, not calibrated',
    'no_chunk': 'no CSV row for this run and time (e.g. bad run), not calibrated',
}
acc = df['calib_status'].value_counts().reindex(list(MEANING), fill_value=0).rename_axis('calib_status').reset_index(name='entries')
acc['fraction_%'] = (100 * acc['entries'] / len(df)).round(2)
acc['meaning'] = acc['calib_status'].map(MEANING)
assert acc['entries'].sum() == len(df)
acc.style.hide(axis='index').format({'entries': '{:,}', 'fraction_%': '{:.2f}'}).set_properties(subset=['meaning'], **{'text-align': 'left'})

## 8. Output Table

- One row per entry; bucket / position / `calib_time_ns` NaN unless `calib_status` = `in_spill`; save with e.g. `calib_table.to_csv(...)`

In [ ]:
calib_table = df[['run', 'subrun', 'event', 'slice_id', 'tdc_rwm_ns', 'uncalib_time_ns', 'calib_chunk', 'calib_bucket',
                  'calib_bucket_pos_ns', 'calib_time_ns', 'calib_status']].copy()
ok = calib_table['calib_status'] == 'in_spill'
assert np.allclose(calib_table['uncalib_time_ns'][ok], (calib_table['calib_bucket_pos_ns'] + calib_table['calib_time_ns'])[ok])
calib_table

### Stacked Bunch per Segment

- `calib_time_ns` of the in-spill entries of each segment (CSV `segment` of the chunk; unstable segment and unsegmented chunks left out); asym Gaussian in ±2.5σ

In [ ]:
calib_table['segment'] = calib_table['calib_chunk'].map(calib['segment'])   # CSV `segment` of the chunk (1, 2, ...), NaN = none
segs = [s for s in ev.SEGMENT_NUMBER if s not in ev.UNSTABLE_SEGMENTS]
fig, axes = plt.subplots(2, 2, figsize=(13.2, 8.8))
for ax, seg in zip(axes.flat, segs):
    num = ev.SEGMENT_NUMBER[seg]
    m = ok & (calib_table['segment'] == num)
    f = stacked_bunch_fit(calib_table.loc[m, 'calib_time_ns'].values, calib.loc[fitted & (calib['segment'] == num), 'period'].median())
    ax.step(f['mid'], f['n'], where='mid', color='k', lw=0.8)
    ax.axvspan(f['mid'][f['mask']][0], f['mid'][f['mask']][-1], color='tab:grey', alpha=0.1)
    draw_asym_fit(ax, f, lambda side, sig, err: rf'$\sigma_{side}$ = {sig:.3f} ± {err:.3f} ns')
    ax.plot([], [], ' ', label=rf"$\chi^2$/ndof = {f['chi_a']:.1f}")
    ax.set(xlabel='calib_time_ns [ns]', ylabel=f"Entries / [{f['bw']:.2f} ns]")
    ax.set_title(f'{seg} (segment {num}): {m.sum()} entries', color=SEG_COLORS[seg], fontweight='bold')
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Full Spill per Segment

- Per segment: `calib_spill_time_ns` = T0 + `calib_bucket` · P + `calib_time_ns` (P = median `period`, T0 = median `intercept` = bucket 0, over the segment's chunks)
- Left `uncalib_time_ns`, right `calib_spill_time_ns`, same entries and 2 ns bins

In [ ]:
fig, axes = plt.subplots(len(segs), 2, figsize=(13.2, 3.3 * len(segs)), sharey='row')
for row_ax, seg in zip(axes, segs):
    num = ev.SEGMENT_NUMBER[seg]
    m = ok & (calib_table['segment'] == num)
    seg_rows = fitted & (calib['segment'] == num)
    P, T0 = calib.loc[seg_rows, 'period'].median(), calib.loc[seg_rows, 'intercept'].median()
    calib_table.loc[m, 'calib_spill_time_ns'] = T0 + calib_table.loc[m, 'calib_bucket'].astype(float) * P + calib_table.loc[m, 'calib_time_ns']
    bins = np.arange(calib_table.loc[m, 'calib_spill_time_ns'].min() - 50, calib_table.loc[m, 'calib_spill_time_ns'].max() + 50, 2)
    for ax, (col, c, title) in zip(row_ax, (('uncalib_time_ns', 'tab:gray', 'Uncalibrated'), ('calib_spill_time_ns', 'tab:blue', 'Calibrated'))):
        ax.hist(calib_table.loc[m, col], bins=bins, histtype='step', color=c, label=col)
        ax.set_title(f'{seg} (segment {num}): {title}', color=SEG_COLORS[seg], fontweight='bold')
        ax.legend(loc='upper center', fontsize=9)
    row_ax[0].set_ylabel('Entries / [2 ns]')
for ax in axes[-1]:
    ax.set_xlabel('Time [ns]')
plt.tight_layout()
plt.show()